# Splitting Audio Samples into Chunks

by Tobias Erbacher

In this notebook, we take the lecture recordings (audio track only) and split them into 2-minute pieces with a rolling window of overlap set to 1 minute, i.e. the first sample will contain minute 0 - 2, the second sample will contain minute 1 - 3, etc. Note that this notebook was developed locally to save on computational costs and due to privacy considerations, for the latter reason, the audio files will not be uploaded to github. Moreover, we do not check whether the actual audio is spoken by the professor, or e.g. questions asked by the students, and therefore the model will inevitably not be completely accurate.

In [1]:
from pydub import AudioSegment
from pydub.silence import detect_nonsilent
import os

INPUT_AUDIO_1 = "audio_files/converted_audio/V.1.1 - Variational Autoencoders.mp3"
INPUT_AUDIO_2 = "audio_files/converted_audio/V.2.1 - The Time Dimension- from Linear to Deep Predictors (first part).mp3"
INPUT_AUDIO_3 = "audio_files/converted_audio/V.6.1 - Leaning in Nonstationary Environments.mp3"

FILES_DICT = {
    INPUT_AUDIO_1 : "f1",
    INPUT_AUDIO_2 : "f2",
    INPUT_AUDIO_3 : "f3",
}

AUDIO_FILES = [INPUT_AUDIO_1, INPUT_AUDIO_2, INPUT_AUDIO_3]
AUDIO_OUTPUT_DIRECTORY = "audio_files/samples_1min"

SAMPLE_LENGTH = 1 # Minutes
OVERLAP = 0.5 # Minutes

# For normalization
TARGET_SAMPLING_RATE = 16000 # Hz
TARGET_CHANNELS = 1 # Mono
TARGET_DBFS = -20.0

# For silence trimming
SILENCE_THRESHOLD = -40
MIN_SILENCE_LENGTH = 500

def split_mp3(input, output_directory, chunk_length, chunk_overlap):
    SAMPLE_DURATION = int(chunk_length * 60 * 1000)
    SAMPLE_OVERLAP = int(chunk_overlap * 60 * 1000)
    STEP_SIZE = SAMPLE_DURATION - SAMPLE_OVERLAP
    os.makedirs(output_directory, exist_ok=True)

    for file in input:
        audio = AudioSegment.from_mp3(file)
        change_dBFS = TARGET_DBFS - audio.dBFS
        audio.apply_gain(change_dBFS)
        audio_length = len(audio)

        c = 1
        for i in range(0, audio_length, STEP_SIZE):
            chunk = audio[i:i + SAMPLE_DURATION]
            nonsilent_ranges = detect_nonsilent(chunk, min_silence_len=MIN_SILENCE_LENGTH, silence_thresh=SILENCE_THRESHOLD)
            if nonsilent_ranges:
                start_trim = nonsilent_ranges[0][0]
                end_trim = nonsilent_ranges[-1][1]
                trimmed_audio = chunk[start_trim:end_trim]
            else:
                trimmed_audio = chunk
            trimmed_audio.set_frame_rate(TARGET_SAMPLING_RATE)
            trimmed_audio.set_channels(TARGET_CHANNELS)
            trimmed_audio_name = os.path.join(output_directory, FILES_DICT[file] + "_sample_" + str(i // chunk_length + 1) + ".wav")
            trimmed_audio.export(trimmed_audio_name, format="wav")
            print("Saved chunk " + trimmed_audio_name)
            c += 1

split_mp3(AUDIO_FILES, AUDIO_OUTPUT_DIRECTORY, SAMPLE_LENGTH, OVERLAP)


Saved chunk audio_files/samples_1min/f1_sample_1.wav
Saved chunk audio_files/samples_1min/f1_sample_30001.wav
Saved chunk audio_files/samples_1min/f1_sample_60001.wav
Saved chunk audio_files/samples_1min/f1_sample_90001.wav
Saved chunk audio_files/samples_1min/f1_sample_120001.wav
Saved chunk audio_files/samples_1min/f1_sample_150001.wav
Saved chunk audio_files/samples_1min/f1_sample_180001.wav
Saved chunk audio_files/samples_1min/f1_sample_210001.wav
Saved chunk audio_files/samples_1min/f1_sample_240001.wav
Saved chunk audio_files/samples_1min/f1_sample_270001.wav
Saved chunk audio_files/samples_1min/f1_sample_300001.wav
Saved chunk audio_files/samples_1min/f1_sample_330001.wav
Saved chunk audio_files/samples_1min/f1_sample_360001.wav
Saved chunk audio_files/samples_1min/f1_sample_390001.wav
Saved chunk audio_files/samples_1min/f1_sample_420001.wav
Saved chunk audio_files/samples_1min/f1_sample_450001.wav
Saved chunk audio_files/samples_1min/f1_sample_480001.wav
Saved chunk audio_file

Next, we infer the text spoken in each chunk with the openai-whisper library to generate the labels that we will later use to train the voice model.

In [3]:
import whisper

TRANSCRIPTION_PATH = AUDIO_OUTPUT_DIRECTORY

OPENAI_MODEL = whisper.load_model("base")

for file in os.listdir(TRANSCRIPTION_PATH):
    if file.endswith(".wav"):
        file_path = os.path.join(TRANSCRIPTION_PATH, file)
        result = OPENAI_MODEL.transcribe(file_path)
        text = result["text"]

        text_file = os.path.splitext(file)[0] + ".txt"
        output_file_path = os.path.join(TRANSCRIPTION_PATH, text_file)

        with open(output_file_path, "w", encoding="utf-8") as f:
            f.write(text)
            print("Transcription saved to " + output_file_path)

/opt/anaconda3/lib/python3.11/site-packages/whisper/transcribe.py:126: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Transcription saved to audio_files/samples_1min/f3_sample_1920001.txt


/opt/anaconda3/lib/python3.11/site-packages/whisper/transcribe.py:126: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Transcription saved to audio_files/samples_1min/f3_sample_930001.txt


/opt/anaconda3/lib/python3.11/site-packages/whisper/transcribe.py:126: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


...

Transcription saved to audio_files/samples_1min/f1_sample_570001.txt


/opt/anaconda3/lib/python3.11/site-packages/whisper/transcribe.py:126: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Transcription saved to audio_files/samples_1min/f2_sample_4110001.txt


/opt/anaconda3/lib/python3.11/site-packages/whisper/transcribe.py:126: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


Transcription saved to audio_files/samples_1min/f1_sample_4860001.txt
